# IIP314W Optimización Aplicada a Negocios - 2026-T1
## Ayudantía 5: Big M y Variables de Activación
### PAUTA DE SOLUCIONES

### Ejercicio 1: Localización de Plantas con Restricciones de Activación

#### Modelamiento Completo

**Conjuntos:**
- $I = \{1, 2, ..., i_{max}\}$: conjunto de plantas potenciales
- $J = \{1, 2, ..., j_{max}\}$: conjunto de ciudades clientes

**Parámetros:**
- $u_i \in \mathbb{R}^+$: capacidad máxima de planta $i$
- $f_i \in \mathbb{R}^+$: costo fijo anual de abrir planta $i$
- $c_{ij} \in \mathbb{R}^+$: costo variable de transporte unitario desde planta $i$ a ciudad $j$
- $d_j \in \mathbb{R}^+$: demanda de ciudad $j$
- $\alpha = 0.30$: factor mínimo de uso de capacidad (30%)
- $K = 2$: máximo número de plantas que pueden estar simultáneamente abiertas
- $M \in \mathbb{R}^+$: parámetro Big M suficientemente grande

**Variables de Decisión:**

- $y_i \in \{0, 1\}$ para cada $i \in I$: variable binaria de **activación**
  - $y_i = 1$ si la planta $i$ está abierta
  - $y_i = 0$ si la planta $i$ está cerrada
  
- $x_{ij} \in \mathbb{R}_+$ para cada $i \in I, j \in J$: cantidad transportada desde planta $i$ a ciudad $j$

**Función Objetivo:**
$$\min Z = \sum_{i \in I} f_i y_i + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij}$$

Minimiza: costo fijo de apertura + costo variable de transporte

**Restricciones:**

**(1) Satisfacción de Demanda (cada ciudad debe ser abastecida):**
$$\sum_{i \in I} x_{ij} = d_j \quad \forall j \in J$$

**(2) Capacidad Condicional a Activación:**
$$\sum_{j \in J} x_{ij} \leq u_i y_i \quad \forall i \in I$$

**(3) Mínimo de Uso (si abierta, debe operar al menos al 30% de capacidad):**
$$\sum_{j \in J} x_{ij} \geq \alpha u_i y_i \quad \forall i \in I$$

O equivalentemente, con Big M:
$$\sum_{j \in J} x_{ij} \geq \alpha u_i - M(1 - y_i) \quad \forall i \in I$$

**Justificación técnica con Big M:**
- Si $y_i = 1$: $\sum_j x_{ij} \geq \alpha u_i - 0 = \alpha u_i$ (debe operar al mínimo)
- Si $y_i = 0$: $\sum_j x_{ij} \geq \alpha u_i - M$ (sin restricción efectiva si $M \geq \alpha u_i$)
- Para este problema, $M = \max_i \{\alpha u_i\}$ es suficiente

**(4) Límite de Plantas Simultáneamente Abiertas:**
$$\sum_{i \in I} y_i \leq K = 2$$

**Justificación:** Restricción lineal sobre variables binarias (sin Big M necesario)

**(5) No-negatividad:**
$$x_{ij} \geq 0 \quad \forall i \in I, j \in J$$
$$y_i \in \{0, 1\} \quad \forall i \in I$$

**Modelo Completo (Forma Estándar):**

$$\begin{align}
\min Z &= \sum_{i \in I} f_i y_i + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij} \\
\text{s.a.} \\
\sum_{i \in I} x_{ij} &= d_j \quad \forall j \in J \\
\sum_{j \in J} x_{ij} &\leq u_i y_i \quad \forall i \in I \\
\sum_{j \in J} x_{ij} &\geq \alpha u_i y_i \quad \forall i \in I \\
\sum_{i \in I} y_i &\leq 2 \\
x_{ij} &\geq 0 \quad \forall i, j \\
y_i &\in \{0, 1\} \quad \forall i
\end{align}$$

**Tipo de Problema:** Programación Lineal Mixta Entera (MIP)

---
### Ejercicio 2: Implementación Simple en Gurobi

#### Solución Completa con Gurobi

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np

# ============================================================================
# 1. DEFINICIÓN DE DATOS
# ============================================================================

# Nombre de plantas y ciudades
plantas = ['Planta_A', 'Planta_B', 'Planta_C']
ciudades = [1, 2, 3, 4]

# Capacidades de plantas (unidades)
capacidad = {
    'Planta_A': 150,
    'Planta_B': 200,
    'Planta_C': 180
}

# Costos fijos de apertura (pesos)
costo_fijo = {
    'Planta_A': 500,
    'Planta_B': 700,
    'Planta_C': 600
}

# Demanda de ciudades (unidades)
demanda = {1: 80, 2: 100, 3: 90, 4: 70}

# Matriz de costos de transporte (pesos/unidad)
# Filas: plantas, Columnas: ciudades
costo_transporte = {
    ('Planta_A', 1): 12, ('Planta_A', 2): 15, ('Planta_A', 3): 18, ('Planta_A', 4): 20,
    ('Planta_B', 1): 14, ('Planta_B', 2): 10, ('Planta_B', 3): 12, ('Planta_B', 4): 16,
    ('Planta_C', 1): 16, ('Planta_C', 2): 14, ('Planta_C', 3): 8,  ('Planta_C', 4): 11,
}

# Parámetros
max_plantas_abiertas = 2
demanda_total = sum(demanda.values())
capacidad_total = sum(capacidad.values())

print("="*70)
print("PROBLEM DATA SUMMARY")
print("="*70)
print(f"Número de plantas: {len(plantas)}")
print(f"Número de ciudades: {len(ciudades)}")
print(f"Demanda total: {demanda_total} unidades")
print(f"Capacidad total disponible: {capacidad_total} unidades")
print(f"Máximo de plantas a abrir: {max_plantas_abiertas}")
print()

In [ ]:
# ============================================================================
# 2. CREACIÓN DEL MODELO EN GUROBI
# ============================================================================

m = gp.Model("Facility_Location_Problem")

# Variables de decisión binarias: ¿está abierta la planta i?
y = {}
for p in plantas:
    y[p] = m.addVar(vtype=GRB.BINARY, name=f"abierta_{p}")

# Variables de decisión continuas: cantidad transportada de planta i a ciudad j
x = {}
for p in plantas:
    for c in ciudades:
        x[p, c] = m.addVar(lb=0, name=f"transport_{p}_to_{c}")

print(f"Variables creadas:")
print(f"  - Variables binarias de activación: {len(y)}")
print(f"  - Variables continuas de transporte: {len(x)}")
print(f"  - Total de variables: {len(y) + len(x)}")
print()

In [ ]:
# ============================================================================
# 3. FUNCIÓN OBJETIVO
# ============================================================================

# Costo fijo total (apertura de plantas)
costo_fijo_total = gp.quicksum(costo_fijo[p] * y[p] for p in plantas)

# Costo variable total (transporte)
costo_variable_total = gp.quicksum(costo_transporte[p, c] * x[p, c] 
                                   for p in plantas for c in ciudades)

# Función objetivo: minimizar costo total
m.setObjective(costo_fijo_total + costo_variable_total, GRB.MINIMIZE)

print("Función Objetivo:")
print("  minimize: Costo_Fijo + Costo_Transporte")
print()

In [ ]:
# ============================================================================
# 4. RESTRICCIONES
# ============================================================================

# Restricción 1: Satisfacción de Demanda
# Cada ciudad debe recibir exactamente su demanda
for c in ciudades:
    m.addConstr(
        gp.quicksum(x[p, c] for p in plantas) == demanda[c],
        name=f"demanda_ciudad_{c}"
    )

print(f"✓ Restricción 1 (Demanda): {len(ciudades)} restricciones agregadas")

# Restricción 2: Capacidad Condicional a Activación
# El flujo saliente de planta i no puede exceder su capacidad si está abierta
# Si y_i = 0 (cerrada): suma_j(x_ij) <= 0, força x_ij = 0
# Si y_i = 1 (abierta): suma_j(x_ij) <= capacidad_i
for p in plantas:
    m.addConstr(
        gp.quicksum(x[p, c] for c in ciudades) <= capacidad[p] * y[p],
        name=f"capacidad_{p}"
    )

print(f"✓ Restricción 2 (Capacidad Condicional): {len(plantas)} restricciones agregadas")

# Restricción 3: Máximo de Plantas Abiertas Simultáneamente
# Por limitaciones energéticas, se pueden abrir a lo más K plantas
m.addConstr(
    gp.quicksum(y[p] for p in plantas) <= max_plantas_abiertas,
    name="max_plantas_abiertas"
)

print(f"✓ Restricción 3 (Máx Plantas): 1 restricción agregada")
print()

print(f"Total de restricciones generadas: {m.numConstrs}")
print()

In [ ]:
# ============================================================================
# 5. OPTIMIZACIÓN
# ============================================================================

print("="*70)
print("OPTIMIZANDO...")
print("="*70)
m.optimize()
print()

# Verificar estado
if m.status == GRB.OPTIMAL:
    print("✓ SOLUCIÓN ÓPTIMA ENCONTRADA")
else:
    print(f"Estado de optimización: {m.status}")
    
print()

In [ ]:
# ============================================================================
# 6. ANÁLISIS DE RESULTADOS
# ============================================================================

print("="*70)
print("RESULTADOS DE LA SOLUCIÓN ÓPTIMA")
print("="*70)
print()

# A) DECISIÓN DE APERTURA DE PLANTAS (Variable de Activación)
print("(A) DECISIÓN DE APERTURA DE PLANTAS (Variables Binarias):")
print("-" * 70)
plantas_abiertas = []
costo_fijo_incurrido = 0

for p in plantas:
    if y[p].X == 1 or y[p].X > 0.99:  # Consideramos 1 si X > 0.99 (tolerancia numérica)
        estado = "ABIERTA"
        plantas_abiertas.append(p)
        costo_fijo_incurrido += costo_fijo[p]
        print(f"  {p:12s}: {estado:8s} ✓ (Costo fijo: ${costo_fijo[p]:,})")
    else:
        estado = "CERRADA"
        print(f"  {p:12s}: {estado:8s} (Costo fijo: $0)")

print(f"\n  Total de plantas abiertas: {len(plantas_abiertas)}/{len(plantas)}")
print(f"  Costo fijo total incurrido: ${costo_fijo_incurrido:,}")
print()

# B) PLAN DE TRANSPORTE ÓPTIMO
print("(B) PLAN DE TRANSPORTE ÓPTIMO:")
print("-" * 70)
transporte_data = []
costo_variable_incurrido = 0

for p in plantas:
    total_desde_planta = 0
    for c in ciudades:
        cantidad = x[p, c].X
        if cantidad > 0.01:  # Solo mostrar flujos significativos
            costo_unit = costo_transporte[p, c]
            costo_por_arco = cantidad * costo_unit
            costo_variable_incurrido += costo_por_arco
            total_desde_planta += cantidad
            transporte_data.append({
                'Origen': p,
                'Destino': f'Ciudad {c}',
                'Cantidad': f'{cantidad:.1f}',
                'Costo/unid': f'${costo_unit}',
                'Costo Total': f'${costo_por_arco:,.0f}'
            })
            print(f"  {p:12s} → Ciudad {c}: {cantidad:6.1f} unid × ${costo_unit:3}/unid = ${costo_por_arco:8,.0f}")
    
    if total_desde_planta > 0:
        cap_utilizada = (total_desde_planta / capacidad[p]) * 100
        print(f"    ├─ Total desde {p}: {total_desde_planta:.1f}/{capacidad[p]} unidades ({cap_utilizada:.1f}% capacidad)")
        print()

print(f"  Costo variable total (transporte): ${costo_variable_incurrido:,}")
print()

In [ ]:
# C) VERIFICACIÓN DE DEMANDA
print("(C) VERIFICACIÓN: DEMANDA SATISFECHA")
print("-" * 70)
demanda_verificacion = []

for c in ciudades:
    total_recibido = sum(x[p, c].X for p in plantas)
    demanda_esperada = demanda[c]
    status = "✓" if abs(total_recibido - demanda_esperada) < 0.01 else "✗"
    print(f"  Ciudad {c}: {total_recibido:6.1f} / {demanda_esperada:3.0f} unidades {status}")
    demanda_verificacion.append({
        'Ciudad': c,
        'Recibido': f'{total_recibido:.1f}',
        'Demanda': demanda_esperada,
        'Status': 'OK' if abs(total_recibido - demanda_esperada) < 0.01 else 'ERROR'
    })

print()

# D) VERIFICACIÓN DE CAPACIDADES
print("(D) VERIFICACIÓN: CAPACIDAD RESPETADA")
print("-" * 70)
capacidad_verificacion = []

for p in plantas:
    total_enviado = sum(x[p, c].X for c in ciudades)
    cap_disponible = capacidad[p] * y[p].X  # Solo si está abierta
    status = "✓" if total_enviado <= cap_disponible + 0.01 else "✗"
    print(f"  {p:12s}: {total_enviado:6.1f} / {cap_disponible:6.1f} unidades {status}")
    capacidad_verificacion.append({
        'Planta': p,
        'Enviado': f'{total_enviado:.1f}',
        'Disponible': f'{cap_disponible:.1f}',
        'Status': 'OK' if total_enviado <= cap_disponible + 0.01 else 'ERROR'
    })

print()

In [ ]:
# E) DESGLOSE DE COSTOS
print("="*70)
print("DESGLOSE DE COSTOS")
print("="*70)

print(f"\nCosto Fijo (Apertura de plantas):")
for p in plantas:
    if y[p].X > 0.99:
        print(f"  {p:12s}: ${costo_fijo[p]:7,}")
print(f"  {'─'*30}")
print(f"  {'COSTO FIJO TOTAL':30s}: ${costo_fijo_incurrido:7,}")

print(f"\nCosto Variable (Transporte):")
print(f"  {'COSTO VARIABLE TOTAL':30s}: ${costo_variable_incurrido:7,.0f}")

print(f"\n{'='*40}")
print(f"  {'COSTO TOTAL MÍNIMO':30s}: ${m.objVal:7,.0f}")
print(f"{'='*40}")

# Porcentajes
pct_fijo = (costo_fijo_incurrido / m.objVal) * 100
pct_variable = (costo_variable_incurrido / m.objVal) * 100

print(f"\nProporción de costos:")
print(f"  Fijo:    {pct_fijo:5.1f}%")
print(f"  Variable: {pct_variable:5.1f}%")
print()

In [ ]:
# F) RESUMEN EJECUTIVO
print("="*70)
print("RESUMEN EJECUTIVO")
print("="*70)

print(f"\n✓ Valor Óptimo de Función Objetivo: ${m.objVal:,.0f}")

print(f"\n✓ Plantas a Abrir:")
for p in plantas_abiertas:
    cap_util = sum(x[p, c].X for c in ciudades)
    print(f"    - {p} (Capacidad utilizada: {cap_util:.0f}/{capacidad[p]} = {(cap_util/capacidad[p])*100:.1f}%)")

print(f"\n✓ Estructuras de Transporte: {len(transporte_data)} rutas activas")

print(f"\n✓ Demanda satisfecha: {sum(demanda.values())}/{demanda_total} unidades (100%)")

print(f"\n✓ Restricción de máximo plantas: {len(plantas_abiertas)}/{max_plantas_abiertas} plantas abiertas")

---
## ANÁLISIS E INTERPRETACIÓN DE RESULTADOS

### Conceptos Clave Aplicados

**1. Variable de Activación ($y_i \in \{0,1\}$)**
- Modela la decisión sí/no de abrir una planta
- Captura el costo fijo asociado a la apertura
- Controla dinámicamente la capacidad disponible

**2. Capacidad Condicional**
- Restricción: $\sum_j x_{ij} \leq u_i y_i$
- Si $y_i=0$: capacidad efectiva es 0 (planta no funciona)
- Si $y_i=1$: capacidad efectiva es $u_i$ (capacidad total)
- No requiere Big M porque el multiplicador es exacto

**3. Restricción de Límite de Plantas**
- $\sum_i y_i \leq 2$ limita simultáneamente a 2 plantas
- Modela restricción de infraestructura/energía

### Interpretación Económica

- El costo fijo de apertura es amortizado por el volumen transportado
- Solo abre plantas si el costo fijo es compensado por la reducción en costos de transporte
- La solución equilibra costos fijos vs variables bajo restricciones operacionales

### Extensiones Posibles

1. Agregar restricción de mínimo uso: $\sum_j x_{ij} \geq 0.30 u_i y_i$ (requiere Big M)
2. Permitir satisfacción parcial de demanda (con penalización)
3. Modelar múltiples períodos de tiempo
4. Considerar inventario en plantas/ciudades